In [ ]:
# ============ IMPORTS ============
import pandas as pd
import numpy as np
import optuna
import mlflow
from src.triple_barrier import triple_barrier
import optuna.visualization as vis
from scipy.stats import entropy as scipy_entropy

# ========== CONFIG ==========
window = 7  # days
min_label_entropy = 0.85
optuna_trials = 100
W = 8  # Used to resize plots, not working.
H = 4
tp_apply_min_ret = False
tp_min_ret = 0.05
# ============================

# ============ LOAD CACHED DATA ============
engineered_features_cache = pd.read_pickle('data_cache/02_engineered_features.pkl')
bitcoin_price_and_features = engineered_features_cache['bitcoin_price_and_features']
bitcoin_features = engineered_features_cache['bitcoin_features']

barrier_optimise_df = bitcoin_features.copy()
barrier_optimise_df['Close'] = bitcoin_price_and_features['Close']

# Use only the TRAINING portion (80%) for the Optuna search.
split_idx   = int(len(barrier_optimise_df) * 0.8)
barrier_optimise_df   = barrier_optimise_df.iloc[:split_idx]

# ============ OBJECTIVE FUNCTION ============
# This is the function Optuna calls once per trial.
# It receives a `trial` object, asks it for parameter suggestions,
# runs the code with those parameters, and returns a score.

def objective(trial):
    # Step 1: Optuna suggests parameter values
    profit_mult = trial.suggest_float("profit_mult", 0.5, 3, step=0.05)
    stop_mult   = trial.suggest_float("stop_mult",   0.5, 3, step=0.05)

    # Step 2: Run triple_barrier with those parameters
    triple_barrier_df = triple_barrier(
        price_series = barrier_optimise_df['Close'],
        volatility_series = barrier_optimise_df['Volatility_EWMA'],
        holding_period = window,          # keep consistent with your notebook
        profit_mult = profit_mult,
        stop_mult = stop_mult,
        min_ret_threshold = tp_min_ret, # Currently not being used, uncomment code in src/triple_barrier.py if needed
        apply_min_ret_threshold = tp_apply_min_ret
    )

    # Step 3: Extract labels and drop NaN rows
    labels = triple_barrier_df['labels'].dropna().astype(int)

    # Step 4: Guard against imbalanced label distributions
    # .reindex([-1, 0, 1], fill_value=0) makes sure counts always has all 3 classes
    # in all 3 runs, even if a class didn't appear. 
    # Missing class proportion (fill_value) = 0
    counts = labels.value_counts(normalize=True).reindex([-1, 0, 1], fill_value=0)
    pos_frequency  = counts[1]
    
    # scipy_entropy(counts) finds Shannon entropy: H = -sum(p * log(p))
    # Dividing by log(3) normalises H to the range [0, 1]:
    label_entropy = scipy_entropy(counts) / np.log(3)

    # These appear in study.trials_dataframe() and in MLflow automatically.
    # counts.max() picks the highest value from [-1, 0, +1] proportions.
    trial.set_user_attr('label_entropy', round(float(label_entropy), 4))
    trial.set_user_attr('lwr_barrier_pct', round(float(counts[-1]), 4))
    trial.set_user_attr('vert_barrier_pct', round(float(counts[0]),  4))
    trial.set_user_attr('upr_barrier_pct', round(float(counts[1]),  4))
    trial.set_user_attr('max_class_pct', round(float(counts.max()), 4))
    
    if label_entropy < min_label_entropy: # Give worst possible score to avoid
        return 0.0
    
    # Mean return of +1, how profitable are +1 labels
    positive_returns = triple_barrier_df.loc[triple_barrier_df['labels'] == 1, 'returns'].dropna()
    avg_pos_return = positive_returns.mean()

    trial.set_user_attr('avg_positive_return', round(float(avg_pos_return), 4))
    trial.set_user_attr('pos_frequency',       round(float(pos_frequency),  4))
    
    return float(pos_frequency * avg_pos_return)

In [ ]:
# ============ OPTUNA SET UP AND RUN ============
# TPESampler uses Bayesian optimisation to focus trials on promising regions.
study = optuna.create_study(
    direction = "maximize", # maximise expected return per bar
    sampler = optuna.samplers.TPESampler(seed=42)  # continuous search, reproducible
)

study.optimize(objective, n_trials = optuna_trials)

best_param = study.best_params
best_trial = study.best_trial
all_values = [trial.value for trial in study.trials if trial.value is not None]
triple_barrier_trials_data = study.trials_dataframe().set_index('number')
triple_barrier_trials_data = pd.DataFrame(triple_barrier_trials_data)
triple_barrier_trials_data.to_pickle("data_cache/03_triple_barrier_trials_data.pkl")
print(f"Best profit_mult: {best_param['profit_mult']}")
print(f"Best stop_mult: {best_param['stop_mult']}")
print(f"Best expected value: {study.best_value:.4f}")

In [ ]:
# PLOT VISUALISING AND EDITING
# plots = {
#     "optimisation_history": vis.plot_optimization_history(study),
#     "contour": vis.plot_contour(study, target_name="EV (+1 Label)"),
#     "slice": vis.plot_slice(study),
#     "param_importances": vis.plot_param_importances(study),
#     "parallel_coordinate": vis.plot_parallel_coordinate(study),
#     "rank": vis.plot_rank(study),
#     "edf": vis.plot_edf(study),
#     "timeline": vis.plot_timeline(study),
#     "contour_entropy"      : vis.plot_contour(study, target=lambda t: t.user_attrs.get("label_entropy", float("nan")),
#                                 target_name="Label Entropy"
#                                 ),
#     "contour_upr_barrier"  : vis.plot_contour(study, target=lambda t: t.user_attrs.get("upr_barrier_pct", float("nan")),
#                                 target_name="Upper Barrier %"
#                                 ),
#     "contour_lwr_barrier"  : vis.plot_contour(study, target=lambda t: t.user_attrs.get("lwr_barrier_pct", float("nan")),
#                                 target_name="Lower Barrier %"
#                                 )
# }
# for name, fig in plots.items():
#     path = f"data_cache/triple_barrier_{name}.html"
#     fig.write_html(path)
#     imgWidth = int((W / 2.54) * 150)
#     imgHeight = int((H / 2.54) * 150)
#     fig.update_layout(margin=dict(l=15, r=15, t=35, b=15))
#     fig.write_image(f"data_cache/triple_barrier_{name}.png", width=imgWidth, height=imgHeight)

In [ ]:
# =========== MLFLOW SETUP ============
mlflow.set_experiment("03_label_parameter_search")

# =========== CALL EXISTING MLFLOW RUNS ============
# experiment  = mlflow.get_experiment_by_name("03_label_parameter_search")
# experiment_id = experiment.experiment_id

# # =========== LIST PREVIOUS RUNS ============
# tp_runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id],
#                             filter_string = "status = 'FINISHED'", #tags.Hyperpar_optimised_for = 'GT_Score', status = 'FINISHED
#                             order_by = ['start_time DESC'])

# tp_run_id = tp_runs.iloc[0]["run_id"]

In [ ]:
# ============ MLFLOW LOG ============
with mlflow.start_run(run_name="triple_barrier_parameters"): #run_name="triple_barrier_parameters"

    # Best parameters
    mlflow.log_params(study.best_params)
    mlflow.log_param("label entropy threshold", min_label_entropy)

    
    # Best trial decomposed metrics
    mlflow.log_metric("best_expected_return_per_bar", best_trial.value)
    mlflow.log_metric("best_avg_positive_return", best_trial.user_attrs['avg_positive_return'])
    mlflow.log_metric("best_pos_frequency", best_trial.user_attrs['pos_frequency'])
    mlflow.log_metric("best_label_entropy", best_trial.user_attrs['label_entropy'])
    mlflow.log_metric("best_lwr_barrier_pct", best_trial.user_attrs['lwr_barrier_pct'])
    mlflow.log_metric("best_vert_barrier_pct", best_trial.user_attrs['vert_barrier_pct'])
    mlflow.log_metric("best_upr_barrier_pct", best_trial.user_attrs['upr_barrier_pct'])
    
    # Study-level metrics
    mlflow.log_metric("mean_expected_return_per_bar", float(np.mean(all_values)))
    mlflow.log_metric("std_expected_return_per_bar", float(np.std(all_values)))
    mlflow.log_metric("n_trials", len(study.trials))
    mlflow.log_metric("n_failed_trials", sum(1 for t in study.trials if t.value is None))

    # All the trials data
    mlflow.log_artifact("data_cache/03_triple_barrier_trials_data.pkl")
    
    # All visualisations as interactive HTML
    plots = {
        "optimisation_history": vis.plot_optimization_history(study),
        "contour": vis.plot_contour(study, target_name="EV(+1 Label)"),
        "slice": vis.plot_slice(study),
        "param_importances": vis.plot_param_importances(study),
        "parallel_coordinate": vis.plot_parallel_coordinate(study),
        "rank": vis.plot_rank(study),
        "edf": vis.plot_edf(study),
        "timeline": vis.plot_timeline(study),
        "contour_entropy"      : vis.plot_contour(study, target=lambda t: t.user_attrs.get("label_entropy", float("nan")),
                                    target_name="Label Entropy"
                                    ),
        "contour_upr_barrier"  : vis.plot_contour(study, target=lambda t: t.user_attrs.get("upr_barrier_pct", float("nan")),
                                    target_name="Upper Barrier %"
                                    ),
        "contour_lwr_barrier"  : vis.plot_contour(study, target=lambda t: t.user_attrs.get("lwr_barrier_pct", float("nan")),
                                    target_name="Lower Barrier %"
                                    )
    }
    for name, fig in plots.items():
        path = f"data_cache/triple_barrier_{name}.html"
        imgPath = f"data_cache/triple_barrier_{name}.png"
        fig.write_html(path)
        imgWidth = int((W / 2.54) * 150)
        imgHeight = int((H / 2.54) * 150)
        fig.update_layout(margin=dict(l=15, r=15, t=35, b=15))
        fig.write_image(imgPath, width=imgWidth, height=imgHeight)
        mlflow.log_artifact(path)
        mlflow.log_artifact(imgPath)